# Anomaly Detection in Industrial Control Systems
In this demo, we analyse the challenges related to anomaly detection in Industrial Control Systems (ICSs). The anomaly detection system is based on a neural network trained on the normal behaviour of the ICS, with the aim to spot anomalies when the actual behaviour diverge from the network's predictions.
The model’s input comprises sensor readings and actuator states collected over the past 10 seconds. Its output is a prediction of the readings for three sensors for the next second. The mean squared error (MSE) between the predicted and actual sensor readings is used to identify potential anomalies in the system.


| <img src="./ics_mlp.jpg" width="90%">  |
|--|
| NN model for anomaly detection|

For the demo, we will rely on the SWaT dataset (https://itrust.sutd.edu.sg/itrust-labs_datasets/dataset_info/) to show the problem of predicting the readings of sensors whose normal behaviour changes over the time.

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from keras.models import Sequential, load_model
from keras.layers import Input, Dense, Dropout, Conv2D, GlobalMaxPooling2D, Flatten,MaxPooling2D
from tensorflow.keras.losses import MeanSquaredError
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.optimizers import Adam,SGD
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from keras.regularizers import l1,l2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix 
from sklearn.metrics import root_mean_squared_error as rmse
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import pandas as pd
import random as rn
import numpy as np
import logging
import time
import pprint
from util_functions_ics import *

# disable GPUs for test reproducibility
tf.config.set_visible_devices([], 'GPU')


MODEL_NAME = "ics_anomaly.keras"
TEST_ITERATIONS = 1

X_train, y_train, y_train_bin= load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5',channels=True)
X_val, y_val, y_val_bin = load_dataset(DATASET_FOLDER + "/*" + '-val.hdf5',channels=True)
X_test, y_test, y_test_bin = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5',channels=True)

In [ ]:
def report_results(Y_true, Y_pred, model_name, data_source, prediction_time):
    ddos_rate = '{:04.3f}'.format(sum(Y_pred) / Y_pred.shape[0])

    if Y_true is not None and len(Y_true.shape) > 0:  # if we have the labels, we can compute the classification accuracy
        Y_true = Y_true.reshape((Y_true.shape[0], 1))
        accuracy = accuracy_score(Y_true, Y_pred)

        f1 = f1_score(Y_true, Y_pred)
        tn, fp, fn, tp = confusion_matrix(Y_true, Y_pred, labels=[0, 1]).ravel()
        tnr = tn / (tn + fp)
        fpr = fp / (fp + tn)
        fnr = fn / (fn + tp)
        tpr = tp / (tp + fn)

        row = {'Model': model_name, 'Time': '{:04.3f}'.format(prediction_time),
               'Samples': Y_pred.shape[0], 'DDOS%': ddos_rate, 'Accuracy': '{:05.4f}'.format(accuracy), 'F1Score': '{:05.4f}'.format(f1),
               'TPR': '{:05.4f}'.format(tpr), 'FPR': '{:05.4f}'.format(fpr), 'TNR': '{:05.4f}'.format(tnr), 'FNR': '{:05.4f}'.format(fnr), 'Source': data_source}

    pprint.pprint(row, sort_dicts=False)

# Concept drift evaluation
We first analyse the behaviour of the sensors' values in the training set

In [ ]:
# Plots of normal readings in the train set

fig, axs = plt.subplots(y_train.shape[1],2)
fig.set_size_inches(18.5, 10.5)
fig.tight_layout(pad=4.0) # space between plots

for col in range(y_train.shape[1]):
    axs[col,0].hist(y_train[:,col], label = "Training set", color = 'orange', edgecolor = 'orange',bins=1000,density=False)
    axs[col,0].set(xlabel='Value', ylabel='Samples')
    axs[col,0].set_title("Readings of sensor: "+SENSORS[col])
    axs[col,0].legend(loc="upper left")

    axs[col,1].plot(y_train[:,col])
    axs[col,1].set(xlabel='Seconds', ylabel='Value')
    axs[col,1].set_title("Readings of sensor: "+SENSORS[col])

# Comparison between training and test sets
Now we compare the sensor readings of training and test sets to highlight possible issues related to concept drift.

In [ ]:
# Plots of normal readings in the test set

fig, axs = plt.subplots(y_test.shape[1],2)
fig.set_size_inches(18.5, 10.5)
fig.tight_layout(pad=4.0) # space between plots

# keep only normal samples
print ("Anomalous samples: ", np.count_nonzero(y_test_bin))
inverted_y_test_bin = np.array(1-y_test_bin)
y_test_normal = np.empty(y_test.shape)
y_test_attack = np.empty(y_test.shape)
for col in range(y_test.shape[1]):
    y_test_normal[:,col] = np.multiply(y_test[:,col],inverted_y_test_bin)
    y_test_normal_no_zeros = y_test_normal[:,col][y_test_normal[:,col] != 0]
    y_test_attack[:,col] = np.multiply(y_test[:,col],y_test_bin)
    y_test_attack_no_zeros = y_test_attack[:,col][y_test_attack[:,col] != 0]

    axs[col,0].hist(y_train[:,col], label="Training set", color = 'orange', edgecolor = 'orange',bins=1000,density=False)
    axs[col,0].hist(y_test_normal_no_zeros, label="Test set (normal readings)", color = 'yellow', edgecolor = 'yellow',bins=1000,density=False)
    axs[col,0].hist(y_test_attack_no_zeros, label="Test set (anomalous readings)", color = 'blue', edgecolor = 'blue',bins=1000,density=False)
    axs[col,0].set(xlabel='Value', ylabel='Samples')
    axs[col,0].set_title("Readings of sensor: "+SENSORS[col])
    axs[col,0].legend(loc="upper left")

    axs[col,1].plot(y_test[:,col])
    axs[col,1].plot(y_test_bin*np.max(y_test[:,col]),color = 'red')
    axs[col,1].set(xlabel='Seconds', ylabel='Value')
    axs[col,1].set_title("Readings of sensor: "+SENSORS[col])

# Model definition
The model is an MLP whose input has the same size of the data samples (X_train.shape[1:]), and the output size is the number of sensors to predict (y_train.shape[1]). Since the data is not normalised, the `linear` activation function is used in the output.

In [ ]:
def compileModel(model,optimizer='sgd', lr=0.001):
    if optimizer == 'sgd':
        optimizer = SGD(learning_rate=lr, momentum=0.0)
    else:
        optimizer = Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999)
    model.compile(loss=MeanSquaredError(), optimizer=optimizer,metrics=['accuracy']) 

In [ ]:
# Function to create the MLP model
def create_MLP(optimizer='adam', dense_layers=2, hidden_units=10, learning_rate = 0.0001, dropout_rate=0, activation='relu'):
    model = Sequential(name  = "mlp")
    model.add(Input(shape=X_train.shape[1:]))
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(dropout_rate))
    for layer in range(dense_layers):
        model.add(Dense(hidden_units, activation='relu', name='hidden-fc' + str(layer)))
        model.add(Dropout(dropout_rate))
    model.add(Dense(y_train.shape[1], activation='linear', name='fc2'))
    compileModel(model, optimizer,learning_rate)
    print (model.summary())
    return model

# Model Training

In [ ]:
PATIENCE = 10 #number of epochs with no improvement on the validation set

# Define early stopping criteria (you can monitor either val_loss or val_accuracy)
early_stopping = EarlyStopping(monitor='val_accuracy', patience=PATIENCE, verbose=1, restore_best_weights=True)

model = create_MLP()
history = model.fit(X_train, y_train, epochs=1000, batch_size=2048, validation_data=(X_val, y_val), verbose=1, callbacks=[early_stopping])

model.save(MODEL_NAME)

# Tune and test the anomaly threshold

## Plot error distribution
Here we compute the detection threshold using the model's error on the validation set.

In [ ]:
# Step 1: analyse the error distrubution
model = load_model(MODEL_NAME)

pt0 = time.time()
for i in range(TEST_ITERATIONS):
    y_pred = model.predict(X_val, batch_size=2048)
pt1 = time.time()
prediction_time = pt1 - pt0

y_pred_diff = abs(y_pred-y_val)

error_mean = np.mean(y_pred_diff,axis=0)
error_std = np.std(y_pred_diff,axis=0)

print ("Average error: ", error_mean)
print ("Max error: ", np.max(y_pred_diff,axis=0))
print ("Min error: ", np.min(y_pred_diff,axis=0))
print ("Std dev error: ", error_std)

# Plot the error distribution for each sensor. The vertical lines represent the mean and the mean plus std_dev
fig, axs = plt.subplots(y_val.shape[1],1)
fig.set_size_inches(18.5, 12.5)
fig.tight_layout(pad=4.0) # space between plots

for col in range(y_pred_diff.shape[1]):
    axs[col].hist(y_pred_diff[:,col], color = 'blue', edgecolor = 'blue',bins=1000,density=False)
    axs[col].set(xlabel='Absolute diff between predicted and ground truth values', ylabel='Samples')
    axs[col].set_title("Readings distribution of sensor: "+SENSORS[col])

    # Vertical lines for the mean and the mean + std dev
    axs[col].axvline(error_mean[col], color='red', linestyle='--', linewidth=2,
                     label='Mean: {:.4f}'.format(error_mean[col]))
    axs[col].axvline(error_mean[col] + error_std[col], color='green', linestyle='--', linewidth=2,
                     label='Mean + Std dev: {:.4f}'.format(error_mean[col] + error_std[col]))
    axs[col].legend(loc='upper right')

plt.show()



## Find the best Z-score  thresholds using the validation set
What is the best Z-score threshold? We can find it by using the validation set, hence, only benign data. For this, we use the ```Elbow method```.

In [ ]:
fpr_list = []
for z_threshold in range(1,11):
    z_scores = (y_pred_diff - error_mean) / error_std

    # Find anomalies on the validation set
    y_pred_bin = np.abs(z_scores) > z_threshold
    y_pred_bin_z_score = np.logical_or(y_pred_bin[:,0],y_pred_bin[:,1]) # anomaly if one of the values is above the threshold
    tn, fp, fn, tp = confusion_matrix(y_val_bin, y_pred_bin_z_score, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fpr_list.append(fpr)

# plot the FPR for different Z-score thresholds
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), fpr_list, marker='o', linestyle='--', color='b')
plt.title('False Positive Rate (FPR) vs Z-score Threshold')
plt.xlabel('Z-score Threshold')
plt.ylabel('False Positive Rate (FPR)')
plt.xticks(range(1, 11))
plt.grid()

In [ ]:
# Set the right threshold based on the plot above
Z_THRESHOLD = 3
y_pred_bin = np.abs(z_scores) > Z_THRESHOLD
y_pred_bin_z_score = np.logical_or(y_pred_bin[:,0],y_pred_bin[:,1]) 
report_results(y_val_bin, y_pred_bin_z_score, model.name, "Validation set", prediction_time)

## Use the threshold on unseen data (the test set)
Here we simulate the deployment of our anomaly detection system.

In [ ]:
# Predictions on the test set
y_pred_test = model.predict(X_test, batch_size=2048) # predictions on the test set
y_pred_test_diff = abs(y_pred_test-y_test)

### Z-score

In [ ]:
z_scores_test = (y_pred_test_diff - error_mean) / error_std

# Find anomalies on the validation set
y_pred_test_bin = np.abs(z_scores_test) > Z_THRESHOLD
y_pred_test_bin_z_score = np.logical_or(y_pred_test_bin[:,0],y_pred_test_bin[:,1]) # anomaly if one of the values is above the threshold

report_results(y_test_bin, y_pred_test_bin_z_score, model.name, "", prediction_time)

### Compare Z-score anomalies with the ground truth labels

In [ ]:
fig, axs = plt.subplots(2,1)
fig.set_size_inches(18.5, 8.5)
fig.tight_layout(pad=4.0) # space between plots

sensor = 0 # index in SENSORS of the sensor to plot

axs[0].plot(y_test[:,sensor])
axs[0].plot(y_test_bin*np.max(y_test[:,sensor]),color = 'red')
axs[0].set(xlabel='Seconds', ylabel='Value')
axs[0].set_title("Readings of sensor: "+SENSORS[sensor])

axs[1].plot(y_test[:,sensor])
axs[1].plot(y_pred_test_bin_z_score*np.max(y_test[:,sensor]),color = 'red')
axs[1].set(xlabel='Seconds', ylabel='Value')
axs[1].set_title("Z-score anomalies on sensor: "+SENSORS[sensor])